
### Dataset: E-Commerce Orders


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

np.random.seed(0)
n = 300

df = pd.DataFrame({
    'order_id': range(1, n+1),
    'customer_id': np.random.randint(1000, 1100, n),
    'age': np.random.randint(18, 60, n),
    'gender': np.random.choice(['Male', 'Female'], n),
    'region': np.random.choice(['Northern Province', 'Southern Province', 'Eastern Province', 'Western Province','Central Province','Copperbelt Province','Muchinga Province','Lusaka Province','Luapula Province'], n),
    'category': np.random.choice(['Electronics', 'Clothing', 'Food', 'Books'], n),
    'quantity': np.random.randint(1, 8, n),
    'unit_price': np.round(np.random.uniform(10, 300, n), 2),
    'discount': np.random.choice([0, 0.1, 0.2, 0.3], n),
    'rating': np.random.randint(1, 6, n),
    'returned': np.random.choice([0, 1], n, p=[0.85, 0.15]),
    'date': pd.date_range('2023-01-01', periods=n, freq='1D')
})

df['revenue'] = np.round(df['quantity'] * df['unit_price'] * (1 - df['discount']), 2)
df['month'] = df['date'].dt.month_name()
df['month_num'] = df['date'].dt.month

print('Dataset ready! Shape:', df.shape)
df.head()

---
## Descriptive Analytics
**Question: What happened?**
> Summarize the data — totals, averages, counts, distributions.

In [ ]:
# Total revenue
print(f'Total Revenue: ${df["revenue"].sum():,.2f}')
print(f'Total Orders: {len(df)}')
print(f'Average Order Value: ${df["revenue"].mean():.2f}')

# Revenue by category
rev_cat = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
print('\nRevenue by Category:')
print(rev_cat)

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(rev_cat.index, rev_cat.values, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
ax.set_title('Total Revenue by Category')
ax.set_ylabel('Revenue ($)')
ax.set_xlabel('Category')
plt.tight_layout()
plt.show()

### 🏋️ Your Exercise
1. Print total revenue per **region**
2. Find the **most common category** ordered
3. Calculate **average rating** per category
4. Plot a **horizontal bar chart** of revenue by region

Hint: `ax.barh(...)` for horizontal bars

In [ ]:
# Your code here!

# 1. Revenue per region

# 2. Most common category

# 3. Avg rating per category

# 4. Horizontal bar chart


---
## 🔍 Type 2: Diagnostic Analytics
**Question: Why did it happen?**
> Dig deeper — find correlations, compare groups, spot what caused a trend.

### ✅ Example

In [ ]:
# Why are some orders returned? Check return rate by category
return_rate = df.groupby('category')['returned'].mean() * 100
print('Return Rate (%) by Category:')
print(return_rate.round(1))

# Does low rating correlate with returns?
print('\nAvg Rating — Returned vs Not Returned:')
print(df.groupby('returned')['rating'].mean().round(2))

# Correlation heatmap
fig, ax = plt.subplots(figsize=(6, 4))
corr = df[['quantity', 'unit_price', 'discount', 'rating', 'returned', 'revenue']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

### 🏋️ Your Exercise
1. Check if **discount level** affects revenue (groupby discount, mean revenue)
2. Compare **average revenue** between Male and Female customers
3. Find which **region has the highest return rate**
4. Plot a **boxplot** of revenue by category to see spread

Hint: `sns.boxplot(data=df, x='category', y='revenue')`

In [ ]:
# Your code here!

# 1. Discount vs revenue

# 2. Revenue by gender

# 3. Return rate by region

# 4. Boxplot


---
## 🔮 Type 3: Predictive Analytics
**Question: What will happen?**
> Use a model to predict future values based on past patterns.

### ✅ Example — Predict Revenue

In [ ]:
from sklearn.preprocessing import LabelEncoder

df2 = df.copy()
le = LabelEncoder()
df2['region_enc'] = le.fit_transform(df2['region'])
df2['category_enc'] = le.fit_transform(df2['category'])
df2['gender_enc'] = le.fit_transform(df2['gender'])

features = ['quantity', 'unit_price', 'discount', 'rating', 'region_enc', 'category_enc']
X = df2[features]
y = df2['revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'R² Score: {r2_score(y_test, y_pred):.3f}')
print('(1.0 = perfect, 0 = no predictive power)\n')

# Plot actual vs predicted
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred, alpha=0.5, color='steelblue')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect Prediction')
ax.set_xlabel('Actual Revenue')
ax.set_ylabel('Predicted Revenue')
ax.set_title('Actual vs Predicted Revenue')
ax.legend()
plt.tight_layout()
plt.show()

### 🏋️ Your Exercise
1. Add `gender_enc` and `age` to the features list and retrain
2. Print the new R² — did it improve?
3. Print each feature and its coefficient (importance)
4. Predict revenue for a **new customer**:
   - quantity=3, unit_price=150, discount=0.1, rating=4, region=North, category=Electronics, gender=Male, age=30

Hint: `model.predict([[3, 150, 0.1, 4, ...]])`

In [ ]:
# Your code here!

# 1. Add gender_enc and age to features

# 2. Retrain and print R²

# 3. Feature coefficients
# for f, c in zip(features, model.coef_):
#     print(f, c)

# 4. Predict for new customer
# new_customer = [[3, 150, 0.1, 4, ...]]
# print('Predicted Revenue:', model.predict(new_customer))


---
## ⚙️ Type 4: Prescriptive Analytics
**Question: What should we do?**
> Turn insights into recommendations and strategies.

### ✅ Example — Find the best strategy

In [ ]:
# Which category + region combo makes most revenue?
pivot = df.pivot_table(values='revenue', index='region', columns='category', aggfunc='sum')
print('Revenue by Region & Category:')
print(pivot.round(0))

# Heatmap of the pivot
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu', ax=ax)
ax.set_title('Revenue Heatmap — Region vs Category')
plt.tight_layout()
plt.show()

# Recommendation
best = pivot.stack().idxmax()
print(f'\n✅ RECOMMENDATION: Focus marketing on {best[1]} in the {best[0]} region — highest revenue potential!')

### 🏋️ Your Exercise
1. Find the **discount level** that produces the highest average revenue — should we increase discounts?
2. Identify the **top 3 customers** (by customer_id) with highest total revenue — target them for loyalty rewards
3. Find categories with **return rate > 15%** — recommend quality improvements
4. Write 3 business recommendations as `print()` statements based on your findings

Example: `print('Recommendation: Reduce discount on Books as it lowers revenue')`

In [ ]:
# Your code here!

# 1. Best discount level

# 2. Top 3 customers

# 3. High return rate categories

# 4. Your 3 recommendations
# print('Recommendation 1: ...')
# print('Recommendation 2: ...')
# print('Recommendation 3: ...')


---
## 🏆 Summary Cheat Sheet
| Type | Question | Python Tools |
|---|---|---|
| Descriptive | What happened? | `describe()`, `groupby()`, bar/pie charts |
| Diagnostic | Why did it happen? | `corr()`, `groupby()`, heatmap, boxplot |
| Predictive | What will happen? | `LinearRegression`, `train_test_split`, R² |
| Prescriptive | What should we do? | `pivot_table()`, heatmap, print recommendations |

**In your hackathon — go through all 4 types in order for maximum impact! 🚀**